In [1]:
import numpy as np

In [2]:
data=[
    "I love this movie",
    "This movie is amazing",
    "I hate this movie",
    "The story was boring",
    "I loved It"
]

<h2>Tokenization</h2>

In [3]:
tokens=[]

In [4]:
for d in data:
    tokens.append(d.lower().split(" "))

In [5]:
tokens

[['i', 'love', 'this', 'movie'],
 ['this', 'movie', 'is', 'amazing'],
 ['i', 'hate', 'this', 'movie'],
 ['the', 'story', 'was', 'boring'],
 ['i', 'loved', 'it']]

<h2>Created Vocabulary Set</h2>

In [6]:
vocabularySet=set()

In [7]:
for data in tokens:
    for d in data:
        vocabularySet.add(d)

In [8]:
vocabularySet

{'amazing',
 'boring',
 'hate',
 'i',
 'is',
 'it',
 'love',
 'loved',
 'movie',
 'story',
 'the',
 'this',
 'was'}

In [9]:
len(vocabularySet)

13

<h2>Vocabulary Indexing</h2>

In [10]:
words_to_id={}

In [11]:
for index,data in enumerate(vocabularySet):
    words_to_id[data]=index+2

In [12]:
words_to_id

{'hate': 2,
 'the': 3,
 'was': 4,
 'love': 5,
 'amazing': 6,
 'i': 7,
 'this': 8,
 'boring': 9,
 'loved': 10,
 'story': 11,
 'is': 12,
 'movie': 13,
 'it': 14}

<h2>Sentence to numbers</h2>

In [13]:
sentence_to_num=[]
for sentence in tokens:
    temp=[]
    for words in sentence:
       temp.append(words_to_id[words])

    sentence_to_num.append(temp)

In [14]:
sentence_to_num

[[7, 5, 8, 13], [8, 13, 12, 6], [7, 2, 8, 13], [3, 11, 4, 9], [7, 10, 14]]

<h2>Padding</h2>

In [15]:
maxLen=len(max(sentence_to_num))

In [16]:
padded=[]
for x in sentence_to_num:
    number_of_zeros=maxLen-len(x)
    padded.append(x+[0]*number_of_zeros)

In [17]:
padded

[[7, 5, 8, 13], [8, 13, 12, 6], [7, 2, 8, 13], [3, 11, 4, 9], [7, 10, 14, 0]]

<h2>Embeddings</h2>

In [18]:
import torch

In [19]:
embedding=torch.nn.Embedding(len(vocabularySet)+2,4,padding_idx=0)

In [20]:
padded_tensor=torch.tensor(padded,dtype=torch.long)

In [21]:
padded_tensor

tensor([[ 7,  5,  8, 13],
        [ 8, 13, 12,  6],
        [ 7,  2,  8, 13],
        [ 3, 11,  4,  9],
        [ 7, 10, 14,  0]])

In [22]:
embedding.weight.shape

torch.Size([15, 4])

In [23]:
embedded=embedding(padded_tensor)

In [24]:
embedded.shape

torch.Size([5, 4, 4])

<h2>Connecting to LSTM layer</h2>

In [25]:
lstm_layer=torch.nn.LSTM(input_size=4,hidden_size=8,batch_first=True)

In [26]:
output_lstm,hiden_lstm=lstm_layer(embedded)

In [27]:
print(f'Output_Shape:{output_lstm.shape}\nHidden:{hiden_lstm[0].shape}\nCell :{hiden_lstm[1].shape}')

Output_Shape:torch.Size([5, 4, 8])
Hidden:torch.Size([1, 5, 8])
Cell :torch.Size([1, 5, 8])


In [28]:
final_hidden_state=hiden_lstm[0].squeeze(0)

In [29]:
final_hidden_state.shape

torch.Size([5, 8])

In [30]:
linear=torch.nn.Linear(8,1)

In [31]:
target=torch.tensor([1,1,0,0,1],dtype=torch.float32)
target=target.reshape(5,-1)

In [32]:
loss_fn=torch.nn.BCEWithLogitsLoss()

In [33]:
optimizer=torch.optim.Adam(list(lstm_layer.parameters())+list(linear.parameters())+list(embedding.parameters()),lr=0.001)

In [34]:
for epoch in range(1001):
    optimizer.zero_grad()

    # Forward pass
    embedded = embedding(padded_tensor)

    output_lstm, (hidden_lstm, cell_lstm) = lstm_layer(embedded)

    final_hidden_state = hidden_lstm[0].squeeze(0)

    prediction = linear(final_hidden_state)

    # Loss
    loss = loss_fn(prediction, target)

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Loss: {loss.item():.4f}")

Loss: 0.7167
Loss: 0.6259
Loss: 0.2736
Loss: 0.0945
Loss: 0.0479
Loss: 0.0304
Loss: 0.0215
Loss: 0.0161
Loss: 0.0127
Loss: 0.0103
Loss: 0.0086


<h2>Input Authentication for testing</h2>

1)Tokenize
2)give appropriant int id if unknow then 1
3) if len of token array shorter then maxlen then pad if more then truncate
4) feed to model to predict

In [35]:
def inputAutheticator(input):
    tempToken=input.lower().split(" ")

    tempSequence=[]
    for word in tempToken:
        if word in words_to_id:
            tempSequence.append(words_to_id[word])
        else:
            tempSequence.append(1)

    paddedTempSequence=[]
    if len(tempSequence)>maxLen:
        paddedTempSequence=tempSequence[0:maxLen]
    elif len(tempSequence)<maxLen:
        paddedTempSequence=(tempSequence+[0]*(maxLen-len(tempSequence)))
    else:
        paddedTempSequence=tempSequence

    return paddedTempSequence


In [83]:
s=inputAutheticator("I love this movie")
s=torch.tensor(s)
s=s.unsqueeze(0)

In [76]:
def prediction(input: list)->bool:
    tempEmbed=embedding(input)
    temp_output_lstm, temp_hidden_lstm = lstm_layer(tempEmbed)
    temp_hidden_state = temp_hidden_lstm[0].squeeze(0)
    temp_prediction = linear(temp_hidden_state)

    return temp_prediction

In [84]:
pOutPut=prediction(s)

In [85]:
val=pOutPut.item()

In [86]:
torch.sigmoid(torch.tensor(val))

tensor(0.9940)